# Vietnamese–English Code-Switching Tagger
**Task:** Given a mixed Vietnamese-English sentence, label every word as **VI** (Vietnamese) or **EN** (English).

**Before running:** click the 📁 Files icon on the left and drag in `train_marked.txt` (and later your CanVEC test file). Then run each cell top to bottom (Shift+Enter).

## 1. The tools
How it works in plain words:
- For each word, we describe it with **features**: its letter chunks (e.g. `ng`, `ươ`, `th` are very Vietnamese; `ck`, `sh`, `tion` are very English), whether it has a Vietnamese tone mark, and the words right before and after it.
- A **logistic regression** model learns from labeled examples which features point to VI and which point to EN.
- To tag a new sentence, it computes features for each word and picks the more likely label.

In [ ]:
import re, random, unicodedata
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


def load_marked(path):
    """Read sentences where English words start with '*'. Returns list of (tokens, labels)."""
    data = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        toks, labs = [], []
        for w in line.split():
            labs.append("EN" if w.startswith("*") else "VI")
            toks.append(w.lstrip("*"))
        data.append((toks, labs))
    return data


def strip_accents(text):
    """Remove Vietnamese tone marks: 'Hôm nay' -> 'Hom nay'. Also đ -> d."""
    text = text.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")


def word_features(tokens, i):
    """Describe word i using its letters and its neighbours."""
    w = tokens[i].lower()
    padded = f"<{w}>"
    f = {"word=" + w: 1, "has_vn_mark": int(strip_accents(w) != w)}
    for n in (1, 2, 3):
        for j in range(len(padded) - n + 1):
            f[f"ng{n}=" + padded[j:j + n]] = 1
    f["prev=" + (tokens[i - 1].lower() if i > 0 else "<START>")] = 1
    f["next=" + (tokens[i + 1].lower() if i < len(tokens) - 1 else "<END>")] = 1
    return f


def to_xy(data):
    X, y = [], []
    for toks, labs in data:
        for i in range(len(toks)):
            X.append(word_features(toks, i))
            y.append(labs[i])
    return X, y


def train(data):
    X, y = to_xy(data)
    model = make_pipeline(DictVectorizer(), LogisticRegression(max_iter=2000, C=5))
    model.fit(X, y)
    return model


def tag(model, sentence):
    toks = re.findall(r"[\wÀ-ỹ']+", sentence)
    preds = model.predict([word_features(toks, i) for i in range(len(toks))])
    return list(zip(toks, preds))


def accuracy(model, data):
    X, y = to_xy(data)
    p = model.predict(X)
    correct = sum(a == b for a, b in zip(p, y))
    return correct / len(y), [(x, a, b) for x, a, b in zip(X, y, p) if a != b]


def no_accent_copy(data):
    return [([strip_accents(t) for t in toks], labs) for toks, labs in data]

## 2. Load the training data
In `train_marked.txt`, English words start with `*`; everything else is Vietnamese.

In [ ]:
train_data = load_marked("train_marked.txt")
print(len(train_data), "training sentences,", sum(len(t) for t, _ in train_data), "words")
n_en = sum(l == "EN" for _, labs in train_data for l in labs)
print("EN words:", n_en, "| VI words:", sum(len(t) for t, _ in train_data) - n_en)
print("Example:", list(zip(*train_data[0])))

## 3. Test sets (never used for training)
**Test set A:** CanVEC sample (real bilingual speech, already labeled). *Loader will be filled in once the file format is checked.*

**Test set B:** your own chat-style sentences **without accents** — the challenging case. Write them in the same `*english` format. ⚠️ The three below are placeholders: replace them with your own ~20 sentences.

In [ ]:
# Test set A: CanVEC (to be connected)
test_canvec = []

# Test set B: YOUR OWN no-accent sentences (replace these placeholders!)
my_sentences = '''
hom nay minh phai lam *homework toi khuya
toi qua *team minh *meeting mai moi xong
mai di an *brunch khong
'''
test_mine = []
for line in my_sentences.strip().splitlines():
    toks = line.split()
    test_mine.append(([w.lstrip("*") for w in toks], ["EN" if w.startswith("*") else "VI" for w in toks]))
print(len(test_mine), "of my own test sentences")

## 4. Baseline model (trained on accented text only)

In [ ]:
baseline = train(train_data)
for name, ts in [("CanVEC", test_canvec), ("My no-accent sentences", test_mine)]:
    if ts:
        acc, errors = accuracy(baseline, ts)
        print(f"Baseline on {name}: {acc:.1%} of words correct, {len(errors)} errors")

## 5. Demo: tag any sentence

In [ ]:
def show(model, sentence):
    print("  ".join(f"{w}/{lab}" for w, lab in tag(model, sentence)))

show(baseline, "Tối nay đi ăn không mình hơi bored")
show(baseline, "hom nay minh co deadline nen phai rush lam")  # no accents: watch for mistakes

## 6. Revision: add no-accent copies of the training data
**Observed problem:** the baseline makes many more mistakes when tone marks are missing (common in real texting).
**Possible cause:** it learned that tone marks ⇒ Vietnamese, so unaccented Vietnamese words like `hom`, `lam` look English.
**One change:** duplicate every training sentence with accents stripped, then retrain and rerun the *same* tests.

In [ ]:
revised = train(train_data + no_accent_copy(train_data))
print(f"{'Test set':28}{'Baseline':>10}{'Revised':>10}")
for name, ts in [("CanVEC", test_canvec), ("My no-accent sentences", test_mine)]:
    if ts:
        print(f"{name:28}{accuracy(baseline, ts)[0]:>10.1%}{accuracy(revised, ts)[0]:>10.1%}")
show(revised, "hom nay minh co deadline nen phai rush lam")

## 7. Remaining errors (use these for your Limitations slide)

In [ ]:
for name, ts in [("CanVEC", test_canvec), ("My no-accent sentences", test_mine)]:
    if ts:
        _, errors = accuracy(revised, ts)
        print(f"--- {name}: {len(errors)} errors")
        for feats, true, pred in errors[:15]:
            word = [k[5:] for k in feats if k.startswith("word=")][0]
            print(f"  {word!r}: expected {true}, got {pred}")